# Assignment 4

In [ ]:
# import libraries
import os
import sys
import sklearn
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import keras
from keras.models import Sequential
from keras.layers import Dense
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Part A: load dataset and create dataframe

In [ ]:
# load the dataset
data = load_breast_cancer()

# extract the features and labels
X = data.data # tumor measurements
y = data.target # tumor labels (0 = malignant, 1 = benign)

feature_names = data.feature_names
target_names = data.target_names

# define the dataframe
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y
df['diagnosis'] = df['target'].apply(lambda x: target_names[x])

# Explanation for the part A

- i loaded the breast cancer dataset using scikit learn function called load_breast_cancer

- next i extracted the components of the scikit-learn dataset using .data, .target, etc to get the input and labels data as well as the feature and label names.

- next i formed the dataframe using pandas function

- also inserted the target column into dataframe using  labels data and mapped the diagnosis usinf label names and labels data

# Part B: exploratory data analysis

In [ ]:
# Part B.3
# print the summary statistics and info
print("Summary Statistics:")
print(df.describe())
print("\n\n\n\nDataFrame Info:")
print(df.info())

# Part B.3 : questions

- How many samples and features are in the dataset?

-- According to dataframe info, there are 569 rows which gives 569 samples and 30 features (29+1 since python starts from 0) and the other 2 columns (target and diagnosis) pertain to labels data.

- Are there any missing values?

-- Looking at the output from df.info() and the non-null counts across all the features and target columns, there does not seem to be any missing values.

- Are features on similar scales or very different?

-- Looking at the df.describe() and the output (e.g. mean, std, etc),  the features are on very different scales. Some features such as area and perimeter have large values while others like smoothness and symmetry are small decimal values. This does show original raw values that the breast cancer dataset was not scaled.

# Code explanation

- For a quick EDA as requested from Part B.3, I utilized dataframe functions such as .describe and .info which provide descriptive statistics, non-null count for each column, and data type of each column.

In [ ]:
# Part B.4 code: create a count plot / bar plot showing how many samples are benign and malignent
sns.countplot(y='diagnosis', data=df, hue='diagnosis')
plt.title('Count of Benign and Malignant Tumors')

# Part B.4 question

- Is the dataset balanced?

-- As can be seen from the figure above, the dataset is not balanced and there are more benign entries (350) compared to malignant (200)


# Code explanation
- To obtain the bar plot, the seaborn's countplot function is used because we only have a single column we want to count and obtain distribution of entries in a single column.

In [ ]:
# part B.5: compute correlation of each feature with target and plot it
df.corr(numeric_only=True)["target"].sort_values().plot(kind='bar')
plt.title('Correlation of Features with Target')
plt.ylabel('Correlation Coefficient')
plt.xlabel('Features')

# Part B.5 questions

- Which 3 features are most positively correlated with target?

-- As can be seen from the plot above, the y axis shows the correlaton coefficient where -1 shows high negative correlaton and +1 shows high positive correlation. There does not seem to be a lot of positively correlated features with correlation coefficient higher than 0.5. The only positively correlated features include smoothness error, mean fractal dimension, texture error with correlation values higher than 0.00 but lower than 0.25 which indicates that these features have positive correlation coefficient but the coefficient is so low that there is no meaningful correlation.



- Which 3 features are most negatively correlated with target?

--  Most positively correlated features with target include worst concave points, worst perimeter, mean concave points which have coreelation coefficient of ~ -0.75


# Code
- i utilized the provided code and added additional labels for axis and title
- the provided code performs correlation between all dataframe columns with target column, sorts the correlation coefficients and plots them in a bar plot

In [ ]:
# Part C

# perform train-test split using 80-20 split and random state of 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# feature scaling using standard scaler (z-score normalization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# build a model using tensorflow and keras
# build at least two models and compare them: model A will be a bseline model with 1 hidden layer and model B will be a deeper model with 2-3 hidden layers

model_A = keras.Sequential([
    keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid') # output layer with sigmoid activation for binary classification
])

model_B = keras.Sequential([
    keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(16, activation='relu'),  
    keras.layers.Dense(1, activation='sigmoid') # output layer with sigmoid activation for binary classification
])

# train the model
model_A.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_A = model_A.fit(X_train_scaled, y_train, epochs=50) # do not set the batch size or validation dataset as per instructor's suggestions

model_B.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_B = model_B.fit(X_train_scaled, y_train, epochs=50)


# evaluate the model on the test dataset and print the test accuracy and confusion matrix
test_loss_A, test_acc_A = model_A.evaluate(X_test_scaled, y_test)
test_loss_B, test_acc_B = model_B.evaluate(X_test_scaled, y_test)

print(f'\n\n\nModel A Test Accuracy: {test_acc_A}')
print(f'Model B Test Accuracy: {test_acc_B}')

# display the confusion matrix for each model
# convert the predicted probabilities to binary labels using a threshold of 0.5
y_pred_A = model_A.predict(X_test_scaled) > 0.5
# compute the confusion matrix
cm_A = confusion_matrix(y_test, y_pred_A)
# create a confusion matrix display object and plot it
disp_A = ConfusionMatrixDisplay(confusion_matrix=cm_A, display_labels=target_names)
disp_A.plot()
plt.title('Confusion Matrix for Model A')

y_pred_B = model_B.predict(X_test_scaled) > 0.5
cm_B = confusion_matrix(y_test, y_pred_B)   
disp_B = ConfusionMatrixDisplay(confusion_matrix=cm_B, display_labels=target_names)
disp_B.plot()
plt.title('Confusion Matrix for Model B')



# Part C explanation

- as can be seen, both models achieved very high test accuracy of 97.37% showing that both neural network architectures are effective at distinguishing between benign and malignant breast cancer cases

- confusion matrices for both models are very similar, with very few misclassifications. This suggests that the models perform well in correctly identifying both classes, although the presence of false negatives indicates that a small number of malignant cases were incorrectly classified as benign, which is an important consideration in medical applications.

- since both models produced the same accuracy and confusion matrix, increasing model complexity by adding additional hidden layers did not lead to improved performance on the test dataset. This suggests that the simpler baseline model is sufficient for this classification task.


# Code
- I performed the train test split using the 80-20 split which is the general split and set the random state of 42 for reproducibility

- next i performed the feature scaling because from EDA we saw that the features have very different scales and i used z-score normalization which centers the mean of each features around 0

- then I tried to create two models of varying complexities using keras library where hidden layers use relu activatoin funciton and last layer uses sigmoid activation function to produce probability output between 0-1

- then i train these models using epoch size of 50. instructor did not recommend setting batch size or validation data.

- then i evaluated the trained model on the test dataset and computed accuracy

- the last step includes computing confusion matrix which first involves defining binary outputs by comparing every value to threshold 0.5 and if the entry is higher than 0.5, the binary output will be 1 and if it is less than 0.5, the binary output will be 0.

- i utilize the scikit-learn functions for computing and displaying the matrix